# 02. Echoes 원곡과 FMA REAL 매칭

앞 단계에서 정리한 Clean TTA 3,162개는 296개의 `original_audio` 그룹으로 구성된다. 이 노트북에서는 `original_audio`의 제목·아티스트 문자열을 FMA metadata와 대조해 각 원곡에 대응하는 `track_id`를 정한다.

여기서는 오디오를 내려받지 않고 metadata만 사용한다. 최종 산출물은 `original_audio`와 FMA REAL track을 연결한 `fma_real_mapping.csv`다.


## 0. 경로 설정

노트북의 실행 위치가 프로젝트 루트 또는 `notebooks/`여도 데이터 경로를 찾을 수 있도록 프로젝트 루트를 확인한다.


In [17]:
from pathlib import Path
import pandas as pd
import numpy as np
import re
import unicodedata

PROJECT_ROOT = Path.cwd()

if not (PROJECT_ROOT / "data").exists():
    if (PROJECT_ROOT.parent / "data").exists():
        PROJECT_ROOT = PROJECT_ROOT.parent

ECHOES_MANIFEST = PROJECT_ROOT / "data/raw/Echoes/Echoes/dataset_manifest.csv"
FMA_TRACKS = PROJECT_ROOT / "data/raw/FMA/fma_metadata/tracks.csv"

print("PROJECT_ROOT:", PROJECT_ROOT)
print("Echoes manifest exists:", ECHOES_MANIFEST.exists())
print("FMA tracks.csv exists:", FMA_TRACKS.exists())


PROJECT_ROOT: <PROJECT_ROOT>
Echoes manifest exists: True
FMA tracks.csv exists: True


**결과:** Echoes manifest와 FMA `tracks.csv`가 지정한 경로에 모두 존재했다.

## 1. Clean TTA 재구성

앞 노트북과 같은 기준으로 TTA만 남기고, 하나의 파일 경로를 여러 행이 공유하는 경우 해당 행을 모두 제외한다. 이후 고유 `original_audio`와 장르를 추린다.


In [18]:
echoes = pd.read_csv(ECHOES_MANIFEST)

tta = echoes[echoes["type"] == "TTA"].copy()

dup_mask = tta["path_in_dataset"].duplicated(keep=False)
tta_clean = tta[~dup_mask].copy()

originals = (
    tta_clean[["original_audio", "genre"]]
    .drop_duplicates("original_audio")
    .sort_values("original_audio")
    .reset_index(drop=True)
)

print("Original TTA rows :", len(tta))
print("Excluded rows     :", int(dup_mask.sum()))
print("Clean TTA rows    :", len(tta_clean))
print("Original groups   :", len(originals))

display(originals.head(10))


Original TTA rows : 3165
Excluded rows     : 3
Clean TTA rows    : 3162
Original groups   : 296


,original_audio,genre
0,"10,000 People Chanting, ""I'm an Individual"" - ...",Electronic
1,1984 - Punk Rock Opera,Rock
2,2 (Wasn't There) - Isle of Pine,Rock
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,Electronic
4,3 am West End - statusq,Electronic
5,5 (Lexington) - Isle of Pine,Rock
6,"50,000 Volts of Democracy mp3 - Legally Blind",Rock
7,"6 (Coat of Arms, Close) - Isle of Pine",Rock
8,A Dark Blue Arc - Pipe Choir,Rock
9,A Different World By Night - Nihilore,Electronic


**결과:** TTA 3,165행에서 경로 충돌 3행을 제외해 3,162행을 남겼다. 고유 `original_audio`는 296개다.


## 2. FMA metadata 정리

FMA의 다중 헤더를 읽은 뒤 `track_id`, 제목, 아티스트, 대표 장르, 라이선스, 재생시간, subset만 별도 표로 만든다.


In [19]:
fma = pd.read_csv(
    FMA_TRACKS,
    header=[0, 1],
    index_col=0
)

fma_simple = pd.DataFrame({
    "track_id": fma.index.astype(int),
    "title": fma[("track", "title")].values,
    "artist": fma[("artist", "name")].values,
    "genre_top": fma[("track", "genre_top")].values,
    "license": fma[("track", "license")].values,
    "duration": fma[("track", "duration")].values,
    "subset": fma[("set", "subset")].values,
})

print("FMA tracks:", len(fma_simple))
display(fma_simple.head())


FMA tracks: 106574


,track_id,title,artist,genre_top,license,duration,subset
0,2,Food,AWOL,Hip-Hop,Attribution-NonCommercial-ShareAlike 3.0 Inter...,168,small
1,3,Electric Ave,AWOL,Hip-Hop,Attribution-NonCommercial-ShareAlike 3.0 Inter...,237,medium
2,5,This World,AWOL,Hip-Hop,Attribution-NonCommercial-ShareAlike 3.0 Inter...,206,small
3,10,Freeway,Kurt Vile,Pop,Attribution-NonCommercial-NoDerivatives (aka M...,161,small
4,20,Spiritual Level,Nicky Cook,NaN,Attribution-NonCommercial-NoDerivatives (aka M...,311,large


**결과:** FMA metadata 106,574곡을 불러왔고, 매칭에 사용할 7개 필드를 `fma_simple`로 정리했다.

## 3. 문자열 정규화

FMA의 제목과 아티스트를 `제목 - 아티스트` 형식으로 합친다. Echoes와 FMA 양쪽 문자열에 NFKC 정규화, 소문자 변환, 앞뒤 및 중복 공백 정리를 적용한다.


In [20]:
def normalize_text(x):
    if pd.isna(x):
        return ""
    x = unicodedata.normalize("NFKC", str(x))
    x = x.strip().lower()
    x = re.sub(r"\s+", " ", x)
    return x

fma_simple["fma_name"] = (
    fma_simple["title"].fillna("").astype(str).str.strip()
    + " - "
    + fma_simple["artist"].fillna("").astype(str).str.strip()
)

fma_simple["match_key"] = fma_simple["fma_name"].map(normalize_text)
originals["match_key"] = originals["original_audio"].map(normalize_text)

display(fma_simple[["track_id","title","artist","fma_name","match_key"]].head())


,track_id,title,artist,fma_name,match_key
0,2,Food,AWOL,Food - AWOL,food - awol
1,3,Electric Ave,AWOL,Electric Ave - AWOL,electric ave - awol
2,5,This World,AWOL,This World - AWOL,this world - awol
3,10,Freeway,Kurt Vile,Freeway - Kurt Vile,freeway - kurt vile
4,20,Spiritual Level,Nicky Cook,Spiritual Level - Nicky Cook,spiritual level - nicky cook


**결과:** FMA 각 행에 원문 조합인 `fma_name`과 비교용 `match_key`가 추가되었다. 출력된 예시에서도 대소문자와 공백이 정리된 키를 확인할 수 있다.

## 4. Exact matching 후보 수

각 Echoes `original_audio`와 같은 `match_key`를 가진 FMA 행의 개수를 센다.


In [21]:
candidate_counts = (
    fma_simple.groupby("match_key")
    .size()
    .rename("candidate_count")
)

match_summary = originals.merge(
    candidate_counts,
    left_on="match_key",
    right_index=True,
    how="left"
)

match_summary["candidate_count"] = (
    match_summary["candidate_count"]
    .fillna(0)
    .astype(int)
)

print("===== EXACT MATCH SUMMARY =====")
print(match_summary["candidate_count"].value_counts().sort_index())

print("\nExactly 1 candidate :", int((match_summary["candidate_count"] == 1).sum()))
print("Multiple candidates :", int((match_summary["candidate_count"] > 1).sum()))
print("No candidate        :", int((match_summary["candidate_count"] == 0).sum()))
print("Total               :", len(match_summary))


===== EXACT MATCH SUMMARY =====
candidate_count
1    279
2     16
5      1
Name: count, dtype: int64

Exactly 1 candidate : 279
Multiple candidates : 17
No candidate        : 0
Total               : 296


**결과:** 296개 원곡 가운데 단일 후보는 279개, 복수 후보는 17개, 무매칭은 0개다. 복수 후보 중 16개 원곡은 후보가 2개이고, `Untitled - Fatal Injection`은 5개다.


## 5. 무매칭 원곡 확인

후보 수가 0인 원곡을 별도로 확인한다.


In [22]:
no_match = match_summary[
    match_summary["candidate_count"] == 0
].copy()

print("No-match count:", len(no_match))
display(no_match[["original_audio", "genre"]])


No-match count: 0


,original_audio,genre


**결과:** 무매칭 원곡은 0개이며 출력 표도 비어 있다.

## 6. 복수 후보 원곡 확인

동일한 제목과 아티스트로 등록된 FMA track이 둘 이상인 원곡을 추린다.


In [23]:
multiple = match_summary[
    match_summary["candidate_count"] > 1
].copy()

print("Multiple-match original_audio count:", len(multiple))
display(multiple[["original_audio", "genre", "candidate_count"]])


Multiple-match original_audio count: 17


,original_audio,genre,candidate_count
1,1984 - Punk Rock Opera,Rock,2
18,"Aquamarine, My Distant Blue - Nihilore",Electronic,2
20,As Nihilism Gives Way To Existentialism - Nihi...,Electronic,2
96,I Know His Blood - Vienna Ditto,Electronic,2
98,I'm gonna try to reach - Los Llamarada,Rock,2
110,KOMFORT - voyageurs,Rock,2
122,Let You're Body Move - D SMILEZ,Electronic,2
130,Lost In The Music (D-Smilez Mix) - D SMILEZ,Electronic,2
133,Loved Ones - Rowan Box,Electronic,2
140,Monkeystage - Ergo Phizmiz,Pop,2


**결과:** 복수 후보가 있는 원곡은 17개다. 16개는 후보가 2개이고 `Untitled - Fatal Injection`만 후보가 5개다.

### 후보 metadata 비교

후보별 `track_id`, 대표 장르, 라이선스, 재생시간과 subset을 함께 확인한다.


In [24]:
multi_candidates = multiple[
    ["original_audio", "genre", "match_key"]
].merge(
    fma_simple[
        [
            "track_id","title","artist","genre_top","license",
            "duration","subset","fma_name","match_key"
        ]
    ],
    on="match_key",
    how="left"
).sort_values(["original_audio", "track_id"])

display(multi_candidates) # Echoes와 FMA 매칭 중 17개의 original_audio가 2개 이상이 후보의 음악을 추출함


,original_audio,genre,match_key,track_id,title,artist,genre_top,license,duration,subset,fma_name
0,1984 - Punk Rock Opera,Rock,1984 - punk rock opera,137212,1984,Punk Rock Opera,Rock,Attribution-NonCommercial,203,small,1984 - Punk Rock Opera
1,1984 - Punk Rock Opera,Rock,1984 - punk rock opera,149410,1984,Punk Rock Opera,Rock,Attribution,200,medium,1984 - Punk Rock Opera
2,"Aquamarine, My Distant Blue - Nihilore",Electronic,"aquamarine, my distant blue - nihilore",134150,"Aquamarine, My Distant Blue",Nihilore,Electronic,Creative Commons Attribution,141,large,"Aquamarine, My Distant Blue - Nihilore"
3,"Aquamarine, My Distant Blue - Nihilore",Electronic,"aquamarine, my distant blue - nihilore",140002,"Aquamarine, My Distant Blue",Nihilore,Electronic,Attribution,141,medium,"Aquamarine, My Distant Blue - Nihilore"
4,As Nihilism Gives Way To Existentialism - Nihi...,Electronic,as nihilism gives way to existentialism - nihi...,134148,As Nihilism Gives Way To Existentialism,Nihilore,Electronic,Creative Commons Attribution,335,large,As Nihilism Gives Way To Existentialism - Nihi...
5,As Nihilism Gives Way To Existentialism - Nihi...,Electronic,as nihilism gives way to existentialism - nihi...,140000,As Nihilism Gives Way To Existentialism,Nihilore,Electronic,Attribution,335,medium,As Nihilism Gives Way To Existentialism - Nihi...
6,I Know His Blood - Vienna Ditto,Electronic,i know his blood - vienna ditto,107616,I Know His Blood,Vienna Ditto,Electronic,Attribution,238,small,I Know His Blood - Vienna Ditto
7,I Know His Blood - Vienna Ditto,Electronic,i know his blood - vienna ditto,147298,I Know His Blood,Vienna Ditto,Electronic,Attribution,238,medium,I Know His Blood - Vienna Ditto
8,I'm gonna try to reach - Los Llamarada,Rock,i'm gonna try to reach - los llamarada,28763,I'm gonna try to reach,Los Llamarada,Rock,Attribution-NoDerivatives 3.0 International,98,large,I'm gonna try to reach - Los Llamarada
9,I'm gonna try to reach - Los Llamarada,Rock,i'm gonna try to reach - los llamarada,84389,I'm Gonna Try To Reach,Los Llamarada,Rock,Attribution-Noncommercial-Share Alike 3.0 Unit...,98,large,I'm Gonna Try To Reach - Los Llamarada


**결과:** 17개 원곡에 해당하는 FMA 후보 37행이 출력되었다. 같은 제목·아티스트라도 `track_id`, 라이선스, 장르 또는 subset이 달라 별도의 선택 기준이 필요하다.

## 7. 단일 후보 매칭

후보가 정확히 하나인 원곡을 FMA metadata와 결합한다.


In [25]:
single = match_summary[
    match_summary["candidate_count"] == 1
][["original_audio", "genre", "match_key"]].copy()

single_matches = single.merge(
    fma_simple[
        [
            "track_id","title","artist","genre_top","license",
            "duration","subset","fma_name","match_key"
        ]
    ],
    on="match_key",
    how="left"
)

print("Single exact matches:", len(single_matches))
display(single_matches.head(20))


Single exact matches: 279


,original_audio,genre,match_key,track_id,title,artist,genre_top,license,duration,subset,fma_name
0,"10,000 People Chanting, ""I'm an Individual"" - ...",Electronic,"10,000 people chanting, ""i'm an individual"" - ...",140932,"10,000 People Chanting, ""I'm an Individual""",Nihilore,Electronic,Creative Commons Attribution,372,medium,"10,000 People Chanting, ""I'm an Individual"" - ..."
1,2 (Wasn't There) - Isle of Pine,Rock,2 (wasn't there) - isle of pine,66449,2 (Wasn't There),Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,108,medium,2 (Wasn't There) - Isle of Pine
2,2Much (Andy Spinelli & Alex Sánchez House Edit...,Electronic,2much (andy spinelli & alex sánchez house edit...,114244,2Much (Andy Spinelli & Alex Sánchez House Edit),Tentacles,Electronic,Attribution,486,medium,2Much (Andy Spinelli & Alex Sánchez House Edit...
3,3 am West End - statusq,Electronic,3 am west end - statusq,112378,3 am West End,statusq,Electronic,Attribution,291,medium,3 am West End - statusq
4,5 (Lexington) - Isle of Pine,Rock,5 (lexington) - isle of pine,66445,5 (Lexington),Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,157,large,5 (Lexington) - Isle of Pine
5,"50,000 Volts of Democracy mp3 - Legally Blind",Rock,"50,000 volts of democracy mp3 - legally blind",130401,"50,000 Volts of Democracy mp3",Legally Blind,Rock,Attribution,270,medium,"50,000 Volts of Democracy mp3 - Legally Blind"
6,"6 (Coat of Arms, Close) - Isle of Pine",Rock,"6 (coat of arms, close) - isle of pine",66446,"6 (Coat of Arms, Close)",Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,204,large,"6 (Coat of Arms, Close) - Isle of Pine"
7,A Dark Blue Arc - Pipe Choir,Rock,a dark blue arc - pipe choir,129963,A Dark Blue Arc,Pipe Choir,Rock,Attribution,327,medium,A Dark Blue Arc - Pipe Choir
8,A Different World By Night - Nihilore,Electronic,a different world by night - nihilore,140926,A Different World By Night,Nihilore,Electronic,Creative Commons Attribution,296,small,A Different World By Night - Nihilore
9,A Lady In Red With A Plan To Steal - Did You J...,Rock,a lady in red with a plan to steal - did you j...,74910,A Lady In Red With A Plan To Steal,Did You Just Hex Me?,Rock,Creative Commons Attribution,134,medium,A Lady In Red With A Plan To Steal - Did You J...


**결과:** 단일 후보 279개가 모두 한 행씩 매칭되었고, 앞의 20개 결과에서 제목·아티스트와 `track_id`를 확인했다.

## 8. 단일 후보의 장르 일치 여부

Echoes `genre`와 FMA `genre_top`을 소문자로 맞춰 비교한다. 장르 값은 후보 검토에만 사용하며, 문자열 매칭 자체의 기준은 아니다.


In [26]:
single_matches["genre_match"] = (
    single_matches["genre"].astype(str).str.lower()
    == single_matches["genre_top"].astype(str).str.lower()
)

print(single_matches["genre_match"].value_counts(dropna=False))

display(
    single_matches.loc[
        ~single_matches["genre_match"],
        ["original_audio","genre","track_id","genre_top","title","artist"]
    ].head(30)
)


genre_match
True    279
Name: count, dtype: int64


,original_audio,genre,track_id,genre_top,title,artist


**결과:** 단일 후보 279개는 Echoes 장르와 FMA 대표 장르가 모두 일치했다. 불일치 표는 비어 있다.

## 9. Exact matching 정리

296개 원곡을 모두 FMA에서 찾았으며, 단일 후보 279개는 바로 연결할 수 있다. 남은 17개 복수 후보에는 라이선스, 장르, `track_id` 순으로 고정된 선택 규칙을 적용한다.


## 10. 원곡 수 재확인

296개가 경로 충돌 제거 때문에 생긴 수치인지 확인하기 위해 Echoes 전체, TTA, ATA, Clean TTA의 고유 `original_audio` 수를 비교한다.


In [27]:
print("전체 Echoes original_audio :", echoes["original_audio"].nunique())

print(
    "TTA original_audio         :",
    echoes.loc[echoes["type"] == "TTA", "original_audio"].nunique()
)

print(
    "ATA original_audio         :",
    echoes.loc[echoes["type"] == "ATA", "original_audio"].nunique()
)

print(
    "Clean TTA original_audio   :",
    tta_clean["original_audio"].nunique()
)

전체 Echoes original_audio : 296
TTA original_audio         : 296
ATA original_audio         : 296
Clean TTA original_audio   : 296


**결과:** 전체 Echoes, TTA, ATA, Clean TTA에서 고유 `original_audio`가 모두 296개였다. 따라서 296은 정제 과정에서 줄어든 값이 아니라 원래 manifest의 원곡 수다.

## 11. 생성기 안의 반복 생성 구조

Suno, Udio, ElevenLabs에서 같은 원곡을 사용한 결과가 몇 개씩 있는지 확인한다.


In [28]:
for gen in ["suno", "udio", "elevenlabs"]:
    temp = tta_clean[tta_clean["generator"] == gen]

    counts = temp["original_audio"].value_counts()
    duplicates = counts[counts > 1]

    print(f"\n===== {gen.upper()} =====")
    print("전체 TTA:", len(temp))
    print("고유 original_audio:", temp["original_audio"].nunique())
    print("중복 original_audio 종류:", len(duplicates))

    print("\n2개 이상 존재하는 original_audio:")
    print(duplicates)


===== SUNO =====
전체 TTA: 300
고유 original_audio: 151
중복 original_audio 종류: 149

2개 이상 존재하는 original_audio:
original_audio
Acoustic Unleashed - Remain                    2
Ad Astra - P C III                             2
All of Us - Eric Skiff                         2
Aquamarine, My Distant Blue - Nihilore         2
Autobahn - S-B-J                               2
                                              ..
Who Loves You Dear - Mink Lungs                2
Windows of the Skull - Sarin                   2
Wir Werden Gott - Japanische Kampfhorspiele    2
Ynkelig - Die Morgendammerung Des Valhalla     2
Новый Нью-Йорк 2 - Чокнутый Пропеллер          2
Name: count, Length: 149, dtype: int64

===== UDIO =====
전체 TTA: 300
고유 original_audio: 151
중복 original_audio 종류: 149

2개 이상 존재하는 original_audio:
original_audio
Acoustic Unleashed - Remain                    2
Ad Astra - P C III                             2
All of Us - Eric Skiff                         2
Aquamarine, My Distant Blue - N

**결과:** Suno와 Udio는 각각 300개 파일에 151개 원곡을 사용했고, 그중 149개 원곡이 두 번 등장한다. ElevenLabs는 150개 원곡을 각각 두 번 사용해 300개 파일을 구성했다.

## 12. 생성기별 구성 비교

12개 생성기에 대해 파일 수, 고유 원곡 수, 원곡당 생성 결과의 최소·최대·평균을 계산한다.


In [29]:
rows = []

for gen in sorted(tta_clean["generator"].unique()):

    temp = tta_clean[
        tta_clean["generator"] == gen
    ]

    counts = temp["original_audio"].value_counts()

    rows.append({
        "generator": gen,
        "total_tta": len(temp),
        "unique_original_audio": temp["original_audio"].nunique(),
        "min_outputs_per_original": counts.min(),
        "max_outputs_per_original": counts.max(),
        "mean_outputs_per_original": counts.mean(),
    })

generator_summary = pd.DataFrame(rows)

display(generator_summary)

,generator,total_tta,unique_original_audio,min_outputs_per_original,max_outputs_per_original,mean_outputs_per_original
0,acestep,294,293,1,2,1.003413
1,audioldm,292,292,1,1,1.000000
2,brev,298,150,1,2,1.986667
3,diffrhythm,299,289,1,3,1.034602
4,elevenlabs,300,150,2,2,2.000000
5,mubert,149,148,1,2,1.006757
6,musicgen,293,293,1,1,1.000000
7,producer,151,150,1,2,1.006667
8,songgen,292,290,1,3,1.006897
9,stableaudio,194,185,1,3,1.048649


**결과:** AudioLDM과 MusicGen은 각 원곡을 한 번씩 사용한 반면, Brev·Suno·Udio는 원곡당 평균 약 1.99개, ElevenLabs는 정확히 2개를 생성했다. 생성기별 고유 원곡 수는 148개에서 293개까지 차이가 난다.

## 13. Suno와 ElevenLabs의 원곡 집합 비교

두 생성기가 사용하는 `original_audio` 집합의 교집합과 합집합을 계산한다.


In [30]:
suno_set = set(
    tta_clean.loc[
        tta_clean["generator"] == "suno",
        "original_audio"
    ]
)

eleven_set = set(
    tta_clean.loc[
        tta_clean["generator"] == "elevenlabs",
        "original_audio"
    ]
)

print("Suno reference:", len(suno_set))
print("ElevenLabs reference:", len(eleven_set))
print("공통 reference:", len(suno_set & eleven_set))
print("두 생성기의 union:", len(suno_set | eleven_set))

Suno reference: 151
ElevenLabs reference: 150
공통 reference: 150
두 생성기의 union: 151


**결과:** Suno는 원곡 151개, ElevenLabs는 150개를 사용한다. 공통 원곡은 150개이고 합집합은 151개이므로 ElevenLabs의 원곡 집합이 Suno 집합에 포함된다.


In [ ]:
for gen in ["suno", "udio", "elevenlabs"]:
    temp = tta_clean[tta_clean["generator"] == gen]

    counts = temp["original_audio"].value_counts()
    duplicates = counts[counts > 1]

    print(f"\n===== {gen.upper()} =====")
    print("전체 TTA:", len(temp))
    print("고유 original_audio:", temp["original_audio"].nunique())
    print("중복 original_audio 종류:", len(duplicates))

    print("\n2개 이상 존재하는 original_audio:")
    print(duplicates)


===== SUNO =====
전체 TTA: 300
고유 original_audio: 151
중복 original_audio 종류: 149

2개 이상 존재하는 original_audio:
original_audio
Acoustic Unleashed - Remain                    2
Ad Astra - P C III                             2
All of Us - Eric Skiff                         2
Aquamarine, My Distant Blue - Nihilore         2
Autobahn - S-B-J                               2
                                              ..
Who Loves You Dear - Mink Lungs                2
Windows of the Skull - Sarin                   2
Wir Werden Gott - Japanische Kampfhorspiele    2
Ynkelig - Die Morgendammerung Des Valhalla     2
Новый Нью-Йорк 2 - Чокнутый Пропеллер          2
Name: count, Length: 149, dtype: int64

===== UDIO =====
전체 TTA: 300
고유 original_audio: 151
중복 original_audio 종류: 149

2개 이상 존재하는 original_audio:
original_audio
Acoustic Unleashed - Remain                    2
Ad Astra - P C III                             2
All of Us - Eric Skiff                         2
Aquamarine, My Distant Blue - N

**결과:** Suno와 Udio는 각각 300개 파일에 151개 원곡을 사용했고, ElevenLabs는 300개 파일에 150개 원곡을 사용했다. 반복 원곡 수는 각각 149개, 149개, 150개였다.

### 생성기별 원곡 사용 방식

Clean TTA 전체에는 원곡 296개가 있지만 모든 생성기가 이를 전부 사용하지는 않는다. 생성기마다 사용하는 원곡 범위와 원곡당 생성 횟수가 다르므로, 생성기별 성능을 비교할 때 표본 구성 차이를 함께 고려한다.


## 원곡 단위 데이터 분할

같은 `original_audio`에서 여러 생성기의 파일이나 동일 생성기의 반복 결과가 나올 수 있다. 파일 단위로 무작위 분할하면 같은 원곡 계열이 학습과 평가 데이터에 함께 들어갈 수 있으므로, 이후 REAL과 FAKE 파일은 `original_audio`를 그룹으로 묶어 같은 split에 배정한다.


## 14. 복수 후보 처리

Exact matching으로 296개 원곡을 모두 찾았지만 17개에는 복수 후보가 있다. 다음 규칙으로 각 원곡의 FMA track을 하나씩 선택한다.


### 14.1 후보 선택 규칙

1. Echoes의 bona-fide 조건에 맞는 CC0, CC-BY 또는 Public Domain 계열 라이선스를 우선한다.
2. 후보가 남으면 Echoes `genre`와 FMA `genre_top`이 같은 행을 우선한다.
3. 조건이 같으면 가장 작은 `track_id`를 고른다.
4. 허용 라이선스 후보가 없으면 장르 일치 여부와 최소 `track_id`로 정한다.

같은 코드를 다시 실행해도 동일한 track이 선택되도록 정렬 순서를 고정한다.


In [33]:
def is_echoes_license(license_value):
    """
    Echoes의 bona-fide 선정 조건에 맞는 라이선스 여부 확인
    """
    if pd.isna(license_value):
        return False

    s = str(license_value).lower()

    # CC0 / Public Domain
    if "cc0" in s or "public domain" in s:
        return True

    # CC-BY 계열
    if "attribution" in s:
        excluded = [
            "noncommercial",
            "non-commercial",
            "no derivatives",
            "noderivatives",
            "sharealike",
            "share alike",
        ]

        if not any(x in s for x in excluded):
            return True

    return False


# --------------------------------------------------
# 모든 FMA 후보 생성
# --------------------------------------------------

fma_candidates = originals[
    ["original_audio", "genre", "match_key"]
].merge(
    fma_simple,
    on="match_key",
    how="left"
)

# 라이선스 조건
fma_candidates["license_allowed"] = (
    fma_candidates["license"].apply(is_echoes_license)
)

# 장르 일치 여부
fma_candidates["genre_match"] = (
    fma_candidates["genre"].astype(str).str.lower()
    ==
    fma_candidates["genre_top"].astype(str).str.lower()
)


# --------------------------------------------------
# original_audio별 최종 후보 선택
# --------------------------------------------------

selected_rows = []

for original_audio, group in fma_candidates.groupby("original_audio"):

    group = group.copy()

    # 후보 개수 기록
    candidate_count = len(group)

    # 1. 라이선스 조건을 만족하는 후보 우선
    allowed = group[group["license_allowed"]]

    if len(allowed) > 0:
        pool = allowed.copy()
        license_fallback = False
    else:
        pool = group.copy()
        license_fallback = True

    # 2. Echoes genre와 FMA genre_top이 일치하는 후보 우선
    genre_matched = pool[pool["genre_match"]]

    if len(genre_matched) > 0:
        pool = genre_matched.copy()

    # 3. 그래도 여러 개면 가장 낮은 track_id 선택
    selected = pool.sort_values("track_id").iloc[0].copy()

    # 그룹 정보를 명시적으로 다시 저장
    selected["original_audio"] = original_audio
    selected["candidate_count"] = candidate_count
    selected["license_fallback"] = license_fallback

    selected_rows.append(selected)


# 최종 DataFrame 생성
selected_real = pd.DataFrame(selected_rows).reset_index(drop=True)


# --------------------------------------------------
# 결과 확인
# --------------------------------------------------

print("===== FINAL FMA REAL MAPPING =====")
print("Original audio :", selected_real["original_audio"].nunique())
print("Selected rows  :", len(selected_real))
print("Selected tracks:", selected_real["track_id"].nunique())

print("\n===== CANDIDATE COUNTS =====")
print(selected_real["candidate_count"].value_counts().sort_index())

print("\n===== LICENSE FALLBACK =====")
print(selected_real["license_fallback"].value_counts())

print("\n===== GENRE MATCH =====")
print(selected_real["genre_match"].value_counts(dropna=False))

===== FINAL FMA REAL MAPPING =====
Original audio : 296
Selected rows  : 296
Selected tracks: 296

===== CANDIDATE COUNTS =====
candidate_count
1    279
2     16
5      1
Name: count, dtype: int64

===== LICENSE FALLBACK =====
license_fallback
False    265
True      31
Name: count, dtype: int64

===== GENRE MATCH =====
genre_match
True    296
Name: count, dtype: int64


**결과:** 296개 원곡에서 서로 다른 FMA track 296개를 선택했다. 후보 수는 1개 279곡, 2개 16곡, 5개 1곡이며, 허용 라이선스가 없어 fallback 규칙을 쓴 원곡은 31개다. 최종 장르는 296곡 모두 Echoes 장르와 일치했다.

### 14.2 복수 후보 17곡 점검

복수 후보에서 선택된 행과 선택 근거를 표로 확인한다.


In [34]:
multiple_selected = selected_real[
    selected_real["candidate_count"] > 1
].copy()

display(
    multiple_selected[
        [
            "original_audio",
            "genre",
            "track_id",
            "genre_top",
            "license",
            "subset",
            "candidate_count",
            "license_allowed",
            "license_fallback",
            "genre_match",
        ]
    ]
)

,original_audio,genre,track_id,genre_top,license,subset,candidate_count,license_allowed,license_fallback,genre_match
1,1984 - Punk Rock Opera,Rock,149410,Rock,Attribution,medium,2,True,False,True
18,"Aquamarine, My Distant Blue - Nihilore",Electronic,134150,Electronic,Creative Commons Attribution,large,2,True,False,True
20,As Nihilism Gives Way To Existentialism - Nihi...,Electronic,134148,Electronic,Creative Commons Attribution,large,2,True,False,True
96,I Know His Blood - Vienna Ditto,Electronic,107616,Electronic,Attribution,small,2,True,False,True
98,I'm gonna try to reach - Los Llamarada,Rock,28763,Rock,Attribution-NoDerivatives 3.0 International,large,2,False,True,True
110,KOMFORT - voyageurs,Rock,42772,Rock,Attribution 3.0 International,medium,2,True,False,True
122,Let You're Body Move - D SMILEZ,Electronic,136017,Electronic,Attribution,large,2,True,False,True
130,Lost In The Music (D-Smilez Mix) - D SMILEZ,Electronic,136016,Electronic,Attribution,large,2,True,False,True
133,Loved Ones - Rowan Box,Electronic,148786,Electronic,Attribution,large,2,True,False,True
140,Monkeystage - Ergo Phizmiz,Pop,22477,Pop,Attribution 3.0 United States,small,2,True,False,True


**결과:** 복수 후보 17곡의 최종 `track_id`가 출력되었다. 이 중 `I'm gonna try to reach`와 `Take Your Fingers` 두 곡은 허용 라이선스 후보가 없어 fallback 규칙을 사용했고, 나머지 15곡은 우선 라이선스 조건을 만족했다.

### 14.3 Description 개수 점검

같은 원곡에 연결된 FAKE 파일 사이에서 description과 장르가 일관적인지 집계한다.


In [35]:
reference_check = (
    tta_clean
    .groupby("original_audio")
    .agg(
        total_fake=("path_in_dataset", "size"),
        generator_count=("generator", "nunique"),
        description_count=("description", "nunique"),
        genre_count=("genre", "nunique")
    )
    .reset_index()
)

print("전체 original_audio:", len(reference_check))

print("\n===== description이 2개 이상인 original_audio =====")

multi_description = reference_check[
    reference_check["description_count"] > 1
].sort_values(
    ["description_count", "original_audio"],
    ascending=[False, True]
)

print("개수:", len(multi_description))

display(multi_description)

전체 original_audio: 296

===== description이 2개 이상인 original_audio =====
개수: 2


,original_audio,total_fake,generator_count,description_count,genre_count
69,"First Glance - Oh Yeah, the Future",14,10,2,1
163,OST 04 Ship under attack - sawsquarenoise,16,12,2,1


**결과:** 296개 원곡 중 description이 둘 이상인 원곡은 2개다. `First Glance`는 FAKE 14개·생성기 10종, `OST 04 Ship under attack`은 FAKE 16개·생성기 12종이며, 두 원곡 모두 장르는 하나로 일치한다.

### 14.4 복수 후보와 라이선스 fallback 비교

복수 후보이거나 fallback 규칙을 사용한 원곡을 한 표로 확인한다.


In [36]:
print("===== Multiple candidate + License fallback =====")

display(
    selected_real[
        (selected_real["candidate_count"] > 1)
        | (selected_real["license_fallback"])
    ][
        [
            "original_audio",
            "genre",
            "track_id",
            "license",
            "candidate_count",
            "license_allowed",
            "license_fallback",
            "genre_match"
        ]
    ].sort_values(
        ["license_fallback", "candidate_count"],
        ascending=[False, False]
    )
)

===== Multiple candidate + License fallback =====


,original_audio,genre,track_id,license,candidate_count,license_allowed,license_fallback,genre_match
98,I'm gonna try to reach - Los Llamarada,Rock,28763,Attribution-NoDerivatives 3.0 International,2,False,True,True
220,Take Your Fingers - Michael Fakesch,Electronic,33594,Attribution-NoDerivatives 3.0 International,2,False,True,True
17,Apparitions Under Glass - Sarin,Rock,113526,Attribution-NoDerivatives 4.0 International,1,False,True,True
40,Coliidae - K.D. Expression,Electronic,33636,Attribution-NoDerivatives 3.0 International,1,False,True,True
44,Crossing - Los Llamarada,Rock,28753,Attribution-NoDerivatives 3.0 International,1,False,True,True
46,Dance of the Martians (SynthStep Edit) - Maxim...,Electronic,136338,Attribution-NoDerivatives 4.0 International,1,False,True,True
47,Daylight Savings - My brother Daniel,Electronic,117280,Attribution-NoDerivatives 4.0 International,1,False,True,True
68,Fifth Ramble - From the album Keith [RSVP007] ...,Electronic,60834,Attribution-NoDerivatives 3.0 International,1,False,True,True
78,Fridge - Railkid Station,Rock,94342,Attribution-NoDerivatives 3.0 International,1,False,True,True
87,"Head of Ancante, Talking Tree - Ak'chamel, The...",Rock,135209,Attribution-NoDerivatives 4.0 International,1,False,True,True


**결과:** 라이선스 fallback 원곡은 31개이며, 복수 후보 17개와 겹치는 원곡은 `I'm gonna try to reach`와 `Take Your Fingers` 두 곡이다. 표의 모든 최종 후보는 Echoes 장르와 일치한다.

### 14.5 Description 상세 확인

Description이 두 개인 원곡의 생성기, 장르, 문자열과 파일 경로를 확인한다.


In [37]:
multi_desc_names = multi_description["original_audio"].tolist()

multi_desc_detail = (
    tta_clean[
        tta_clean["original_audio"].isin(multi_desc_names)
    ][
        [
            "original_audio",
            "generator",
            "genre",
            "description",
            "path_in_dataset"
        ]
    ]
    .sort_values(
        ["original_audio", "description", "generator"]
    )
)

display(multi_desc_detail)

,original_audio,generator,genre,description,path_in_dataset
69,"First Glance - Oh Yeah, the Future",acestep,Pop,No description available,TTA/acestep/First_Glance_Oh_Yeah_the_Future_ac...
995,"First Glance - Oh Yeah, the Future",brev,Pop,No description available,TTA/brev/First_Glance_Oh_Yeah_the_Future_brev_...
996,"First Glance - Oh Yeah, the Future",brev,Pop,No description available,TTA/brev/First_Glance_Oh_Yeah_the_Future_brev_...
1490,"First Glance - Oh Yeah, the Future",diffrhythm,Pop,No description available,TTA/diffrhythm/First_Glance_Oh_Yeah_the_Future...
4015,"First Glance - Oh Yeah, the Future",musicgen,Pop,No description available,TTA/musicgen/First_Glance_Oh_Yeah_the_Future_m...
2039,"First Glance - Oh Yeah, the Future",producer,Pop,No description available,TTA/producer/First_Glance_Oh_Yeah_the_Future_p...
2348,"First Glance - Oh Yeah, the Future",songgen,Pop,No description available,TTA/songgen/First_Glance_Oh_Yeah_the_Future_so...
3438,"First Glance - Oh Yeah, the Future",stableaudio,Pop,No description available,TTA/stableaudio/First_Glance_Oh_Yeah_the_Futur...
2899,"First Glance - Oh Yeah, the Future",suno,Pop,No description available,TTA/suno/First_Glance_Oh_Yeah_the_Future_suno_...
2900,"First Glance - Oh Yeah, the Future",suno,Pop,No description available,TTA/suno/First_Glance_Oh_Yeah_the_Future_suno_...


**결과:** `First Glance`의 일반 기록은 `No description available`이고 ElevenLabs 두 파일에만 구체적인 description이 있다. `OST 04 Ship under attack`은 ElevenLabs 기록에서 `boss-fight` 한 항목만 빠져 있으며, 장르 값은 원곡별로 일관된다.

### 14.6 Description 원문 비교

두 원곡의 고유 문자열과 각 문자열을 사용한 생성기를 원문 그대로 출력한다.


In [38]:
for name in multi_desc_names:
    print("\n" + "=" * 100)
    print(name)
    print("=" * 100)

    temp = tta_clean[
        tta_clean["original_audio"] == name
    ]

    descriptions = temp["description"].drop_duplicates()

    print("Unique descriptions:", len(descriptions))

    for i, desc in enumerate(descriptions, start=1):
        print(f"\n[Description {i}]")
        print(repr(desc))

        gens = (
            temp.loc[
                temp["description"] == desc,
                "generator"
            ]
            .value_counts()
        )

        print("\nGenerators:")
        print(gens)


First Glance - Oh Yeah, the Future
Unique descriptions: 2

[Description 1]
'No description available'

Generators:
generator
brev           2
suno           2
udio           2
acestep        1
diffrhythm     1
producer       1
songgen        1
stableaudio    1
musicgen       1
Name: count, dtype: int64

[Description 2]
'indie-synthpop, dreamy-pads, airy-male-vocal, midtempo, nostalgic, melodic, reverb, gentle, romantic, hazy'

Generators:
generator
elevenlabs    2
Name: count, dtype: int64

OST 04 Ship under attack - sawsquarenoise
Unique descriptions: 2

[Description 1]
'chiptune, frantic-tempo, staccato-arps, alarm-like, tense, energetic, 8-bit, boss-fight, looping, instrumental'

Generators:
generator
brev           2
suno           2
udio           2
acestep        1
audioldm       1
diffrhythm     1
mubert         1
producer       1
songgen        1
stableaudio    1
musicgen       1
Name: count, dtype: int64

[Description 2]
'chiptune, frantic-tempo, staccato-arps, alarm-like, te

**결과:** 두 원곡 모두 description 차이는 ElevenLabs 기록과 나머지 생성기 기록 사이에서 발생했다. `First Glance`는 description 유무의 차이이고, `OST 04 Ship under attack`은 태그 한 개의 차이여서 서로 다른 원곡이 섞인 경우로 보지 않았다.

## 15. FMA REAL 매핑 저장

선택한 296개 track과 후보 수, 라이선스 판정, 장르 일치 여부를 CSV로 저장한다.


In [39]:
from pathlib import Path

output_dir = PROJECT_ROOT / "data/metadata"
output_dir.mkdir(parents=True, exist_ok=True)

mapping_path = output_dir / "fma_real_mapping.csv"

mapping_columns = [
    "original_audio",
    "genre",
    "track_id",
    "title",
    "artist",
    "genre_top",
    "license",
    "duration",
    "subset",
    "candidate_count",
    "license_allowed",
    "license_fallback",
    "genre_match",
]

fma_real_mapping = (
    selected_real[mapping_columns]
    .sort_values("original_audio")
    .reset_index(drop=True)
)

fma_real_mapping.to_csv(
    mapping_path,
    index=False,
    encoding="utf-8-sig"
)

print("Saved:", mapping_path)
print("Rows:", len(fma_real_mapping))
print("Unique original_audio:", fma_real_mapping["original_audio"].nunique())
print("Unique track_id:", fma_real_mapping["track_id"].nunique())

display(fma_real_mapping.head(10))

Saved: <PROJECT_ROOT>/data/metadata/fma_real_mapping.csv
Rows: 296
Unique original_audio: 296
Unique track_id: 296


,original_audio,genre,track_id,title,artist,genre_top,license,duration,subset,candidate_count,license_allowed,license_fallback,genre_match
0,"10,000 People Chanting, ""I'm an Individual"" - ...",Electronic,140932,"10,000 People Chanting, ""I'm an Individual""",Nihilore,Electronic,Creative Commons Attribution,372,medium,1,True,False,True
1,1984 - Punk Rock Opera,Rock,149410,1984,Punk Rock Opera,Rock,Attribution,200,medium,2,True,False,True
2,2 (Wasn't There) - Isle of Pine,Rock,66449,2 (Wasn't There),Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,108,medium,1,True,False,True
3,2Much (Andy Spinelli & Alex Sánchez House Edit...,Electronic,114244,2Much (Andy Spinelli & Alex Sánchez House Edit),Tentacles,Electronic,Attribution,486,medium,1,True,False,True
4,3 am West End - statusq,Electronic,112378,3 am West End,statusq,Electronic,Attribution,291,medium,1,True,False,True
5,5 (Lexington) - Isle of Pine,Rock,66445,5 (Lexington),Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,157,large,1,True,False,True
6,"50,000 Volts of Democracy mp3 - Legally Blind",Rock,130401,"50,000 Volts of Democracy mp3",Legally Blind,Rock,Attribution,270,medium,1,True,False,True
7,"6 (Coat of Arms, Close) - Isle of Pine",Rock,66446,"6 (Coat of Arms, Close)",Isle of Pine,Rock,Attribution-NoDerivs 2.5 Canada,204,large,1,True,False,True
8,A Dark Blue Arc - Pipe Choir,Rock,129963,A Dark Blue Arc,Pipe Choir,Rock,Attribution,327,medium,1,True,False,True
9,A Different World By Night - Nihilore,Electronic,140926,A Different World By Night,Nihilore,Electronic,Creative Commons Attribution,296,small,1,True,False,True


**결과:** `data/metadata/fma_real_mapping.csv`에 296행을 저장했다. `original_audio`와 `track_id`는 각각 296개로 모두 고유하다.

## 16. FMA subset 분포

선택한 REAL track이 FMA의 small, medium, large subset에 몇 곡씩 포함되는지 확인한다.


In [40]:
print("===== FMA SUBSET DISTRIBUTION =====")

print(
    selected_real["subset"]
    .value_counts(dropna=False)
)

print("\nTotal:", len(selected_real))

===== FMA SUBSET DISTRIBUTION =====
subset
small     122
medium    103
large      71
Name: count, dtype: int64

Total: 296


**결과:** 선택한 296곡은 small 122곡, medium 103곡, large 71곡으로 나뉜다.


In [41]:
subset_genre = pd.crosstab(
    selected_real["subset"],
    selected_real["genre"]
)

display(subset_genre)

genre,Electronic,Pop,Rock
subset,,,
large,26,11,34
medium,47,0,56
small,45,56,21


**결과:** subset과 장르의 교차표에서 large는 Electronic 26·Pop 11·Rock 34곡, medium은 Electronic 47·Rock 56곡, small은 Electronic 45·Pop 56·Rock 21곡이다. 각 행의 합은 large 71곡, medium 103곡, small 122곡이다.

## 17. 개별 음원 경로 확인

FMA 전체 archive 대신 필요한 296곡만 확보할 수 있는지 `raw_tracks.csv`의 원본 파일 경로를 확인한다.


In [42]:
RAW_TRACKS = PROJECT_ROOT / "data/raw/FMA/fma_metadata/raw_tracks.csv"

raw_tracks = pd.read_csv(
    RAW_TRACKS,
    index_col=0
)

print("raw_tracks shape:", raw_tracks.shape)

print("\n===== 관련 컬럼 =====")
print([
    col for col in raw_tracks.columns
    if "file" in col.lower() or "duration" in col.lower()
])

raw_tracks shape: (109727, 38)

===== 관련 컬럼 =====
['license_image_file', 'license_image_file_large', 'track_duration', 'track_file', 'track_image_file']


**결과:** `raw_tracks.csv`는 109,727행, 38열이며 원본 파일 위치를 담은 `track_file`과 `track_duration` 컬럼을 확인했다.


In [43]:
selected_ids = selected_real["track_id"].astype(int).tolist()

raw_selected = raw_tracks.loc[
    raw_tracks.index.intersection(selected_ids)
].copy()

print("필요한 track_id:", len(selected_ids))
print("raw_tracks에서 찾은 track_id:", len(raw_selected))

display(
    raw_selected[
        ["track_file", "track_duration"]
    ].head(10)
)

필요한 track_id: 296
raw_tracks에서 찾은 track_id: 296


,track_file,track_duration
track_id,,
1382,music/WFMU/Parsley_Flakes/Parsley_Flakes_mp3s/...,02:10
1881,music/WFMU/The_Tleilaxu_Music_Machine/The_Tlei...,04:13
3836,music/WFMU/Lightning_Bolt/Live_at_WFMU_on_Bria...,05:01
3857,music/WFMU/Los_Fancy_Free/Live_at_WFMU_on_Liz_...,03:17
3936,music/WFMU/Mink_Lungs/Live_at_WFMU_on_Scotts_S...,02:10
3956,music/WFMU/Miss_Derringer/Live_at_WFMU_of_Joe_...,02:38
3959,music/WFMU/Miss_Derringer/Live_at_WFMU_of_Joe_...,02:47
3961,music/WFMU/Mod_Fun/Live_at_WFMU_on_Pat_Duncans...,04:24
3962,music/WFMU/Mod_Fun/Live_at_WFMU_on_Pat_Duncans...,03:52


**결과:** 필요한 `track_id` 296개를 `raw_tracks.csv`에서 모두 찾았다. 출력 표에는 앞의 10개 원본 경로와 재생시간이 제시되어 있다.


In [44]:
raw_selected["download_url"] = (
    "https://files.freemusicarchive.org/"
    + raw_selected["track_file"].astype(str)
)

display(
    raw_selected[
        ["track_file", "track_duration", "download_url"]
    ].head(5)
)

,track_file,track_duration,download_url
track_id,,,
1382,music/WFMU/Parsley_Flakes/Parsley_Flakes_mp3s/...,02:10,https://files.freemusicarchive.org/music/WFMU/...
1881,music/WFMU/The_Tleilaxu_Music_Machine/The_Tlei...,04:13,https://files.freemusicarchive.org/music/WFMU/...
3836,music/WFMU/Lightning_Bolt/Live_at_WFMU_on_Bria...,05:01,https://files.freemusicarchive.org/music/WFMU/...
3857,music/WFMU/Los_Fancy_Free/Live_at_WFMU_on_Liz_...,03:17,https://files.freemusicarchive.org/music/WFMU/...
3936,music/WFMU/Mink_Lungs/Live_at_WFMU_on_Scotts_S...,02:10,https://files.freemusicarchive.org/music/WFMU/...


**결과:** `track_file` 앞에 FMA 파일 서버 주소를 붙여 곡별 다운로드 URL을 만들었고, 앞의 5개 URL을 확인했다.


In [45]:
import requests

HF_DATASET = "benjamin-paine/free-music-archive-large"
HF_API = "https://datasets-server.huggingface.co/filter"

params = {
    "dataset": HF_DATASET,
    "config": "default",
    "split": "train",
    "where": '"title"=\'1984\' AND "artist"=\'Punk Rock Opera\'',
    "length": 10,
}

response = requests.get(
    HF_API,
    params=params,
    timeout=60
)

print("status:", response.status_code)
print("URL:", response.url)

data = response.json()

print("검색 결과 수:", len(data.get("rows", [])))

for result in data.get("rows", []):
    row = result["row"]

    print("\nTitle :", row.get("title"))
    print("Artist:", row.get("artist"))
    print("Audio :", row.get("audio"))

status: 500
URL: https://datasets-server.huggingface.co/filter?dataset=benjamin-paine%2Ffree-music-archive-large&config=default&split=train&where=%22title%22%3D%271984%27+AND+%22artist%22%3D%27Punk+Rock+Opera%27&length=10
검색 결과 수: 0


**결과:** Hugging Face datasets-server에 제목과 아티스트로 시험 요청을 보냈으나 상태 코드 500이 반환되었고 검색 결과는 0개였다. 이 경로로는 개별 파일 조회를 확인하지 못했다.


## 정리

- 296개 원곡: 단일 후보 279개, 복수 후보 17개, 무매칭 0개
- 선택 결과: 고유 FMA track 296개, 장르 일치 296개
- 라이선스 fallback: 31개
- Subset: small 122곡, medium 103곡, large 71곡
- 매핑 파일: `data/metadata/fma_real_mapping.csv`

`raw_tracks.csv`에서는 296개 track의 원본 경로를 모두 찾았다. 다만 마지막 Hugging Face API 시험 요청은 상태 코드 500으로 실패해 검색 결과를 얻지 못했다.
